In [49]:
attr = pd.read_csv('C:/NUS WORK/Y4S1/DSA4213 NLP/ProjectCode/data/processed/attractions.csv')
attr['borough'].unique()

<ArrowStringArray>
['Manhattan', 'Brooklyn', 'Bronx', 'Queens', 'Staten Island', nan]
Length: 6, dtype: str

In [76]:
accoms = pd.read_parquet("C:/NUS WORK/Y4S1/DSA4213 NLP/ProjectCode/data/raw/hotels.parquet")

accoms = accoms.drop(columns=["full_address", "state_full", "type", "state_abbrev", "phone_number","timezone", "website", "tld", "place_link", "reviews_link", "booking_link", "country", "business_id", 'about_details_json', "place_id", 'google_mid', 'cid', 'subtypes', 'business_status', 'verified', 'primary_photo_url', 'primary_photo_url_large', 'photo_count', 'owner_id', 'owner_name', 'data_quality_score', 'data_quality_flags'])


accoms = accoms[accoms['city'].isin (["Bronx", "Queens", "Staten Island", "Manhattan", "Brooklyn"])]

accoms.rename(columns={'city': 'area'}, inplace=True)

accoms.info()
accoms['district'].unique()



<class 'pandas.DataFrame'>
Index: 242 entries, 0 to 1088
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   name                      242 non-null    str    
 1   street_address            242 non-null    str    
 2   district                  238 non-null    str    
 3   area                      242 non-null    str    
 4   zipcode                   242 non-null    str    
 5   latitude                  242 non-null    float64
 6   longitude                 242 non-null    float64
 7   rating                    237 non-null    float64
 8   review_count              6 non-null      float64
 9   hotel_stars               152 non-null    float64
 10  price_usd                 107 non-null    float64
 11  amenity_free_wifi         242 non-null    bool   
 12  amenity_free_breakfast    242 non-null    bool   
 13  amenity_free_parking      242 non-null    bool   
 14  amenity_pool             

<ArrowStringArray>
[               'West Bronx',                'East Bronx',
                   'Melrose',                   'Belmont',
               'Port Morris',            'Morris Heights',
            'Pelham Gardens',                   'Norwood',
                 'Manhattan',              'Williamsburg',
               'Brownsville',              'Borough Park',
                'Greenpoint',               'Boerum Hill',
         'Downtown Brooklyn',              'Bridge Plaza',
                   'Gowanus',                  'Red Hook',
          'Brooklyn Heights',        'Bedford-Stuyvesant',
               'Fort Greene',                  'Bushwick',
               'Sunset Park', 'Prospect Lefferts Gardens',
             'Crown Heights',        'Stuyvesant Heights',
                'Park Slope',          'Little Caribbean',
                  'Flatbush',             'Fort Hamilton',
             'East New York',         'East Williamsburg',
             'Cypress Hills',        

In [ ]:
# Fill the gaps in accoms (the Google parquet rows) with model-based synthetic values.
# Rating is the anchor: it drives hotel_stars, and rating plus stars drive price. The
# observed values are never changed. These are estimates, NOT real data. Seed 42.
accoms = accoms.copy()
rng = np.random.default_rng(42)
has_rating = accoms["rating"].notna()
has_stars = accoms["hotel_stars"].notna()
has_price = accoms["price_usd"].notna()

# district (4 missing): district of the nearest hotel in the same area
for i in accoms.index[accoms["district"].isna()]:
    same = accoms[(accoms["area"] == accoms.at[i, "area"]) & accoms["district"].notna()]
    dist = np.hypot(same["latitude"] - accoms.at[i, "latitude"],
                    same["longitude"] - accoms.at[i, "longitude"])
    accoms.at[i, "district"] = same.loc[dist.idxmin(), "district"]

# rating (5 missing): from hotel_stars where known, otherwise the median of the area
both = has_rating & has_stars
slope, intercept = np.polyfit(accoms.loc[both, "hotel_stars"], accoms.loc[both, "rating"], 1)
area_median = accoms.groupby("area")["rating"].transform("median")
rating_from_stars = intercept + slope * accoms["hotel_stars"]
rating_fill = rating_from_stars.where(has_stars, area_median)
accoms["rating"] = accoms["rating"].fillna(rating_fill).clip(1, 5).round(1)

# hotel_stars (90 missing): linear fit of stars on rating plus noise, 1 decimal place
s_slope, s_intercept = np.polyfit(accoms.loc[both, "rating"], accoms.loc[both, "hotel_stars"], 1)
s_resid = accoms.loc[both, "hotel_stars"] - (s_intercept + s_slope * accoms.loc[both, "rating"])
stars_pred = s_intercept + s_slope * accoms["rating"] + rng.normal(0, 0.5 * s_resid.std(), len(accoms))
accoms["hotel_stars"] = accoms["hotel_stars"].fillna(pd.Series(stars_pred, index=accoms.index).clip(1, 5)).round(1)

# price_usd (135 missing): log-price regression on stars, rating and borough, plus noise
borough_dummies = pd.get_dummies(accoms["area"]).astype(float)
X = np.column_stack([np.ones(len(accoms)), accoms["hotel_stars"], accoms["rating"], borough_dummies.drop(columns="Bronx")])
y = np.log(accoms["price_usd"])
coef, *_ = np.linalg.lstsq(X[has_price], y[has_price], rcond=None)
p_resid = y[has_price] - X[has_price] @ coef
price_pred = np.exp(X @ coef + rng.normal(0, 0.7 * p_resid.std(), len(accoms)))
accoms["price_usd"] = accoms["price_usd"].fillna(pd.Series(price_pred, index=accoms.index).clip(60, 740)).round(0)

# review_count (236 missing): log-normal, higher for higher-star hotels
count_pred = np.exp(rng.normal(5.0 + 0.4 * (accoms["hotel_stars"] - 3), 0.8))
accoms["review_count"] = accoms["review_count"].fillna(pd.Series(count_pred, index=accoms.index).clip(10, 5000)).round(0)

# amenities: keep the observed True flags and add more with the same star-based
# chances used for accoms2 (observed rows have at most 2 flags, so they look incomplete)
for col, (p_low, p_high) in AMENITY_PROB.items():
    p = np.interp(accoms["hotel_stars"], [1, 5], [p_low, p_high])
    accoms[col] = accoms[col] | (rng.random(len(accoms)) < p)

# description (114 missing): sentence built from the row's own columns
AMENITY_LABELS = {c: c.replace("amenity_", "").replace("_", " ") for c in AMENITY_PROB}
def describe(row):
    amenities = [label for col, label in AMENITY_LABELS.items() if row[col]]
    text = (f"{row['name']} is a {row['hotel_stars']:.1f}-star hotel in {row['district']}, "
            f"{row['area']}, rated {row['rating']:.1f} out of 5 by guests.")
    if amenities:
        text += " Amenities include " + ", ".join(amenities) + "."
    return text
missing_description = accoms["description"].isna()
accoms.loc[missing_description, "description"] = accoms[missing_description].apply(describe, axis=1)

print(accoms.isna().sum()[accoms.isna().sum() > 0])
print(f"stars = {s_intercept:.2f} + {s_slope:.2f} * rating; log price coefficients:", coef.round(2))


In [ ]:
accoms2 = pd.read_csv("C:/NUS WORK/Y4S1/DSA4213 NLP/ProjectCode/data/raw/accomodations_manhatten.csv")

CITY_TO_AREA = {
    "New York": "Manhattan",
    "New York City": "Manhattan",
    "Manhattan": "Manhattan",
    "Long Island City": "Queens",
    "Bronx": "Bronx",
}

# Columns that have an equivalent in accoms
accoms2 = accoms2.rename(columns={
    "neighborhood": "district",
    "starLevel": "hotel_stars",
    "overallScore": "rating",
    "reviewCount": "review_count",
})
accoms2["area"] = accoms2["city"].map(CITY_TO_AREA)
assert accoms2["area"].notna().all()

# overallScore is out of 10, accoms rating is out of 5
accoms2["rating"] = accoms2["rating"] / 2

# Keep exactly accoms' columns, in the same order. This drops the columns with no
# equivalent (city, countryCode, ratingLabel, cleanliness, locationRating, staffRating,
# hotelId, scrapedAt, descriptionLanguage, reviews). Columns accoms has but accoms2
# lacks (street_address, zipcode, price_usd, amenity_*) are added empty.
accoms2 = accoms2.reindex(columns=accoms.columns)
amenity_cols = [c for c in accoms.columns if c.startswith("amenity_")]
accoms2[amenity_cols] = accoms2[amenity_cols].astype("boolean")  # unknown, not False
assert list(accoms2.columns) == list(accoms.columns)

accoms2.info()


In [ ]:
import numpy as np

# accoms2 has no street address, zipcode, price or amenities, so fill them with
# synthetic values (seed 42). Price and amenities scale with hotel_stars, so higher
# star hotels cost more and have more amenities. These are NOT real data.
rng = np.random.default_rng(42)

accoms2["hotel_stars"] = accoms2["hotel_stars"].round(1)
stars = accoms2["hotel_stars"].to_numpy()

# price_usd: uniform between a per-star low and high (Manhattan nightly rates)
star_points = [1, 2, 3, 4, 5]
price_low = np.interp(stars, star_points, [70, 100, 150, 230, 400])
price_high = np.interp(stars, star_points, [110, 170, 250, 400, 800])
accoms2["price_usd"] = rng.uniform(price_low, price_high).round(0)

# amenities: chance of each amenity at 1 star and at 5 stars, interpolated by stars
AMENITY_PROB = {
    "amenity_free_wifi": (0.80, 0.98),
    "amenity_free_breakfast": (0.15, 0.40),
    "amenity_free_parking": (0.04, 0.04),
    "amenity_pool": (0.00, 0.25),
    "amenity_fitness_center": (0.30, 0.98),
    "amenity_pet_friendly": (0.25, 0.50),
    "amenity_air_conditioning": (0.80, 1.00),
    "amenity_restaurant": (0.10, 0.90),
    "amenity_bar_lounge": (0.10, 0.90),
    "amenity_business_center": (0.05, 0.80),
    "amenity_airport_shuttle": (0.02, 0.15),
    "amenity_spa": (0.00, 0.50),
    "amenity_ev_charging": (0.00, 0.20),
}
for col, (p_low, p_high) in AMENITY_PROB.items():
    p = np.interp(stars, [1, 5], [p_low, p_high])
    accoms2[col] = rng.random(len(accoms2)) < p

# zipcode and street_address: plausible for each neighborhood
ZIPS = {
    "Times Square - Theatre District": ["10036", "10019"],
    "Midtown West - Hell's Kitchen": ["10019", "10036"],
    "Central Park South": ["10019", "10022"],
    "Madison Square Garden - Convention Area": ["10001", "10119"],
    "Empire State Building Area": ["10001", "10016"],
    "Grand Central Terminal - UN": ["10017"],
    "Midtown East": ["10022", "10017"],
    "Rockefeller Center - Bryant Park": ["10020", "10036"],
    "Chelsea Area": ["10011", "10001"],
    "SoHo - Tribeca": ["10013", "10012"],
    "Downtown - Wall Street": ["10004", "10005", "10006", "10038"],
    "Lower East Side - Chinatown": ["10002", "10013"],
    "Upper West Side": ["10023", "10024", "10025"],
    "Upper East Side": ["10021", "10028", "10075"],
    "Morningside Heights": ["10027"],
    "Washington Heights": ["10032", "10033"],
    "Long Island City": ["11101"],
    "Bronx": ["10451", "10454"],
}
STREETS = {  # (low street, high street); lower Manhattan uses named streets below
    "Times Square - Theatre District": (41, 52),
    "Midtown West - Hell's Kitchen": (42, 56),
    "Central Park South": (55, 59),
    "Madison Square Garden - Convention Area": (30, 37),
    "Empire State Building Area": (28, 38),
    "Grand Central Terminal - UN": (38, 49),
    "Midtown East": (45, 57),
    "Rockefeller Center - Bryant Park": (40, 52),
    "Chelsea Area": (15, 29),
    "Upper West Side": (70, 96),
    "Upper East Side": (65, 92),
    "Morningside Heights": (110, 122),
    "Washington Heights": (155, 181),
}
NAMED = {
    "SoHo - Tribeca": ["Broome St", "Spring St", "Canal St", "Greene St", "Varick St"],
    "Downtown - Wall Street": ["Wall St", "Water St", "Pearl St", "William St", "Broad St"],
    "Lower East Side - Chinatown": ["Orchard St", "Allen St", "Delancey St", "Bowery", "Grand St"],
    "Long Island City": ["Jackson Ave", "Queens Plaza S", "44th Dr"],
    "Bronx": ["Grand Concourse", "E 149th St"],
}

def ordinal(n):
    return f"{n}{'th' if 10 <= n % 100 <= 20 else {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')}"

zipcodes, addresses = [], []
for district in accoms2["district"]:
    zipcodes.append(rng.choice(ZIPS[district]))
    number = int(rng.integers(1, 400))
    if district in NAMED:
        addresses.append(f"{number} {rng.choice(NAMED[district])}")
    else:
        low, high = STREETS[district]
        side = "W" if rng.random() < 0.5 else "E"
        addresses.append(f"{number} {side} {ordinal(int(rng.integers(low, high + 1)))} St")
accoms2["zipcode"] = zipcodes
accoms2["street_address"] = addresses

assert accoms2[["street_address", "zipcode", "price_usd", *AMENITY_PROB]].notna().all().all()
accoms2.info()


In [ ]:
# Drop accoms2 hotels that are already in accoms (same spot, within ~25 m); the accoms
# row has real price/amenities, so it wins.
def metres_to_nearest_accoms(row):
    dist = np.hypot((accoms["latitude"] - row["latitude"]) * 111000,
                    (accoms["longitude"] - row["longitude"]) * 84000)
    return dist.min()

duplicate = accoms2.apply(metres_to_nearest_accoms, axis=1) < 25
print("Dropping duplicates from accoms2:", accoms2.loc[duplicate, "name"].tolist())

accoms_all = pd.concat([accoms, accoms2[~duplicate]], ignore_index=True)

print(accoms_all.shape)
print(accoms_all["area"].value_counts())
accoms_all.info()


In [ ]:
accoms_all.to_csv("C:/NUS WORK/Y4S1/DSA4213 NLP/ProjectCode/data/processed/accomodation.csv", index=False)
